# Module 01: Understanding fMRI Data

This notebook introduces the basic concepts behind functional MRI (fMRI) and shows you how to
work with NIfTI images in Python.

## What is fMRI?

Functional MRI measures brain activity indirectly by detecting changes in blood-oxygen-level-
dependent (BOLD) contrast. When neurons fire they consume oxygen, triggering a localised
increase in blood flow. This produces small (~1–5%) changes in MR signal intensity that can be
detected over time.

Key properties of a typical fMRI acquisition:

| Property | Typical value |
|----------|---------------|
| Voxel size | 2–3 mm isotropic |
| TR (repetition time) | 1–2 s |
| Number of volumes | 100–500 |
| Slices per volume | 30–80 |

The result is a **4-D array** (x, y, z, time) stored in NIfTI format.

## Learning Objectives

1. Understand the NIfTI file format and what it contains.
2. Load and inspect NIfTI images with nibabel.
3. Explore image dimensions, affine matrices, and header metadata.
4. Visualise brain volumes with nilearn.

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("nibabel version :", nib.__version__)
print("numpy version   :", np.__version__)

## 1. The NIfTI File Format

NIfTI (Neuroimaging Informatics Technology Initiative) is the standard file format for MRI data
in research. A NIfTI file contains:

- A **header** (352 bytes for NIfTI-1) storing metadata such as voxel dimensions, data type,
  units, and the affine transformation.
- A **data array** holding the image intensities.

Files are saved either as `.nii` (header + data together) or `.nii.gz` (gzip-compressed).

### The affine matrix

The **4×4 affine matrix** maps voxel indices to real-world (scanner or standard-space) coordinates
in millimetres. Understanding the affine is essential when registering images to a common space.

In [ ]:
# Create a synthetic 3-D anatomical-like image (64×64×40, 2 mm isotropic)
shape_3d = (64, 64, 40)
data_3d = np.random.randn(*shape_3d).astype(np.float32)

# 2 mm isotropic affine, origin at the image centre
vox_size = 2.0
affine = np.array([
    [vox_size, 0,        0,        -shape_3d[0] * vox_size / 2],
    [0,        vox_size, 0,        -shape_3d[1] * vox_size / 2],
    [0,        0,        vox_size, -shape_3d[2] * vox_size / 2],
    [0,        0,        0,         1                          ],
])

img_3d = nib.Nifti1Image(data_3d, affine)

print("Shape  :", img_3d.shape)
print("Dtype  :", img_3d.get_data_dtype())
print("Zooms  :", img_3d.header.get_zooms(), "mm")
print("Affine :")
print(img_3d.affine)

## 2. Creating a Synthetic 4-D fMRI Time Series

Real fMRI data has a 4th dimension representing time. Each volume (3-D frame) is called a
**TR** (repetition time). Below we simulate a short run of 100 volumes.

In [ ]:
n_volumes = 100
TR = 2.0  # seconds

shape_4d = (*shape_3d, n_volumes)
data_4d = np.random.randn(*shape_4d).astype(np.float32)

# Embed a simple sinusoidal signal in a central voxel to mimic BOLD
cx, cy, cz = [s // 2 for s in shape_3d]
t = np.arange(n_volumes) * TR
data_4d[cx, cy, cz, :] += 2.0 * np.sin(2 * np.pi * t / 20)

img_4d = nib.Nifti1Image(data_4d, affine)
img_4d.header.set_zooms((*img_3d.header.get_zooms(), TR))

print("4-D image shape :", img_4d.shape)
print("Zooms (x,y,z,t) :", img_4d.header.get_zooms())
print(f"Total acquisition time: {n_volumes * TR:.0f} s ({n_volumes * TR / 60:.1f} min)")

## 3. Exploring Header Metadata

The NIfTI header stores a wealth of metadata. The most commonly used fields are shown below.

In [ ]:
hdr = img_4d.header

fields_of_interest = [
    "dim",        # array dimensions
    "pixdim",     # voxel sizes (+ TR in dim[4])
    "datatype",   # numeric type code
    "xyzt_units", # spatial + temporal units
    "descrip",    # free-text description
]

print(f"{'Field':<15} Value")
print("-" * 50)
for field in fields_of_interest:
    print(f"{field:<15} {hdr[field]}")

print("\nData shape (from header):", hdr.get_data_shape())
print("Voxel sizes             :", hdr.get_zooms())
print("Slope / intercept       :", hdr.get_slope_inter())

## 4. Visualising Brain Data with nilearn

nilearn's `plotting` module provides quick, publication-quality brain visualisations. We will
use `plot_epi` to display a single fMRI volume and `plot_stat_map` to overlay a synthetic
activation map.

In [ ]:
try:
    from nilearn import image, plotting

    # Extract the first volume from our 4-D image
    first_vol = image.index_img(img_4d, 0)

    fig, axes = plt.subplots(1, 1, figsize=(8, 3))
    display = plotting.plot_epi(
        first_vol,
        title="Synthetic fMRI volume (TR=0)",
        display_mode="ortho",
        axes=axes,
        colorbar=True,
    )
    plt.tight_layout()
    plt.savefig("/tmp/fmri_first_volume.png", dpi=80, bbox_inches="tight")
    plt.close()
    print("✅  Figure saved to /tmp/fmri_first_volume.png")

except ImportError:
    print("⚠️   nilearn not available. Install with: conda install -c conda-forge nilearn")

## 5. Extracting a BOLD Time Series

One of the most common operations in fMRI analysis is extracting the signal from a region of
interest (ROI) and plotting it as a time series.

In [ ]:
bold_signal = img_4d.get_fdata()[cx, cy, cz, :]
time_axis = np.arange(n_volumes) * TR

plt.figure(figsize=(10, 3))
plt.plot(time_axis, bold_signal, lw=1.5, color="steelblue")
plt.xlabel("Time (s)")
plt.ylabel("Signal (a.u.)")
plt.title(f"BOLD time series — central voxel ({cx}, {cy}, {cz})")
plt.tight_layout()
plt.savefig("/tmp/bold_time_series.png", dpi=80, bbox_inches="tight")
plt.close()
print("✅  Time series plot saved to /tmp/bold_time_series.png")

## 6. Using a Real Dataset from nilearn

nilearn bundles several small datasets. `fetch_development_fmri` returns resting-state data from
children and adults — useful for demonstrating real data properties.

In [ ]:
try:
    from nilearn import datasets as nl_datasets

    # Fetch a single subject; data downloads to ~/nilearn_data/ on first run (~30 MB)
    dataset = nl_datasets.fetch_development_fmri(n_subjects=1)
    func_path = dataset.func[0]
    real_img = nib.load(func_path)

    print("File            :", func_path)
    print("Shape           :", real_img.shape)
    print("Voxel size (mm) :", real_img.header.get_zooms()[:3])
    print("TR (s)          :", real_img.header.get_zooms()[3])
    n_trs = real_img.shape[3]
    tr_val = real_img.header.get_zooms()[3]
    print(f"Run duration    : {n_trs} TRs = {n_trs * tr_val:.0f} s")

except ImportError:
    print("⚠️   nilearn not available — skipping real-data demo.")
except Exception as exc:
    print(f"⚠️   Could not fetch dataset: {exc}")

## Summary

In this notebook you have:

- Learned what fMRI measures and how data are represented as 4-D arrays.
- Created and inspected synthetic NIfTI images (3-D and 4-D) with nibabel.
- Explored image shape, voxel sizes, affine matrices, and header fields.
- Visualised fMRI volumes and BOLD time series with nilearn and matplotlib.

### Next Notebook

Continue with **`01_bids_overview.ipynb`** to learn how fMRI datasets are organised using the
BIDS standard.